# Variance of a Sum and a Difference

Why subtracting two random returns does **not** subtract their risk, and how covariance turns that
into diversification, tracking error and hedging.

## 0. Math link

This notebook applies two results from the probability course:

- Linearity of expectation: [Sum of expectations](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/04_sum_of_expectations.ipynb)
- Variance and covariance: [Variance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/05_variance.ipynb),
  [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb)

## 1. Introduction

Expectation is linear, always. Variance is not, and the term that breaks linearity is the
covariance. That one term explains three things practitioners deal with every day:

- **Diversification**: why a portfolio of two volatile assets can be less volatile than either.
- **Tracking error**: why an index fund with 15% volatility can deviate from its benchmark by only 1%.
- **Hedging**: why a hedge only works with strongly correlated instruments, and backfires with independent ones.

After this notebook you can compute the variance of any sum or difference of two returns, and
read what the correlation is doing in each case.

## 2. Theory

### Expectation of a sum

$$
E[aX + bY] = a\,E[X] + b\,E[Y]
$$

No assumptions about how $X$ and $Y$ are related.

### Variance of a sum

Expand the square around the means $\mu_X, \mu_Y$:

$$
\begin{aligned}
\operatorname{Var}(X+Y) &= E\big[(X-\mu_X + Y-\mu_Y)^2\big] \\
&= E[(X-\mu_X)^2] + E[(Y-\mu_Y)^2] + 2\,E[(X-\mu_X)(Y-\mu_Y)] \\
&= \operatorname{Var}(X) + \operatorname{Var}(Y) + 2\operatorname{Cov}(X,Y)
\end{aligned}
$$

### Variance of a difference

Use $\operatorname{Var}(-Y) = (-1)^2 \operatorname{Var}(Y)$ and $\operatorname{Cov}(X,-Y) = -\operatorname{Cov}(X,Y)$:

$$
\operatorname{Var}(X-Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) - 2\operatorname{Cov}(X,Y)
$$

The minus sign is squared away on the variance, and survives only on the covariance.

### General weighted form

With $\operatorname{Cov}(X,Y) = \rho\,\sigma_X\sigma_Y$:

$$
\operatorname{Var}(aX + bY) = a^2\sigma_X^2 + b^2\sigma_Y^2 + 2ab\,\rho\,\sigma_X\sigma_Y
$$

- Sum: $a = b = 1$. Difference: $a = 1, b = -1$. Portfolio: $a = w_1, b = w_2$.
- Independent (or just uncorrelated): $\rho = 0$, variances add in **both** cases.

## 3. Worked Examples

### Example 1: independent variables

$\operatorname{Var}(X) = \operatorname{Var}(Y) = 4$, independent.

$$
\operatorname{Var}(X-Y) = 4 + 4 - 2\cdot 0 = 8
$$

Not 0. Two sources of noise, both still there.

### Example 2: two dice, independent vs glued

One die: $E = 3.5$, $\operatorname{Var} = 35/12 \approx 2.92$.

| | $E[X_1+X_2]$ | $\operatorname{Var}(X_1+X_2)$ |
|---|---|---|
| Independent ($\rho=0$) | 7 | $2 \cdot 2.92 = 5.83$ |
| Glued, same face ($\rho=1$) | 7 | $\operatorname{Var}(2X) = 4 \cdot 2.92 = 11.67$ |

Same expectation, double the variance.

### Example 3: the correlation that makes a difference riskless

With $\sigma_X = \sigma_Y = \sigma$:

$$
\operatorname{Var}(X-Y) = 2\sigma^2(1-\rho)
$$

This is zero exactly when $\rho = 1$, and largest ($4\sigma^2$) when $\rho = -1$.

## 4. Finance Application

### Diversification (sum)

Equal weights in two assets, each with 20% annual volatility:

$$
\sigma_p^2 = 0.5^2(0.2^2) + 0.5^2(0.2^2) + 2(0.5)(0.5)\rho(0.2)(0.2) = 0.02\,(1+\rho)
$$

| $\rho$ | 1.0 | 0.5 | 0.0 | −0.5 |
|---|---|---|---|---|
| $\sigma_p$ | 20.0% | 17.3% | 14.1% | 10.0% |

Diversification wants **low** correlation.

### Tracking error (difference)

Active return $R_F - R_B$ (fund minus benchmark). Tracking error is its standard deviation:

$$
TE = \sqrt{\sigma_F^2 + \sigma_B^2 - 2\rho\,\sigma_F\sigma_B}
$$

Fund 15%, benchmark 14%, $\rho = 0.98$: $TE = \sqrt{0.0225 + 0.0196 - 0.04116} \approx 3.1\%$.
Two volatile return streams, a small difference, all thanks to the covariance term.

### Hedging (difference)

Long exposure $X$, short hedge $Y$, both 20% volatility:

| Hedge correlation $\rho$ | 0.0 | 0.9 | 0.99 |
|---|---|---|---|
| $\sigma(X-Y)$ | 28.3% | 8.9% | 2.8% |

A hedge with an **uncorrelated** instrument raises risk from 20% to 28.3%. With equal
volatilities, $2\sigma^2(1-\rho) < \sigma^2$ only when $\rho > 0.5$: below that, a 1:1 hedge
adds risk. (Module 03/03 shows how to choose the hedge ratio instead of fixing it at 1.)

### The usual misreading

"Subtracting the benchmark (or the hedge) removes its risk." It removes only the part that
co-moves. Differences want **high** correlation, sums want **low** correlation. Same formula,
opposite sign on the covariance term.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

### Check Example 1 by simulation

In [ ]:
n = 1_000_000
x = rng.normal(loc=0, scale=2, size=n)   # Var = 4
y = rng.normal(loc=0, scale=2, size=n)   # Var = 4, independent of x

print(f"Var(X - Y) simulated: {np.var(x - y):.3f}   (theory: 8)")
print(f"Var(X + Y) simulated: {np.var(x + y):.3f}   (theory: 8)")

### Two dice: independent vs glued

In [ ]:
die_1 = rng.integers(1, 7, size=n)
die_2 = rng.integers(1, 7, size=n)

independent_sum = die_1 + die_2
glued_sum = 2 * die_1

summary = pd.DataFrame(
    {
        "mean": [independent_sum.mean(), glued_sum.mean()],
        "variance": [independent_sum.var(), glued_sum.var()],
        "theory variance": [2 * 35 / 12, 4 * 35 / 12],
    },
    index=["independent", "glued"],
)
summary.round(3)

### A helper for any two-variable linear combination

In [ ]:
def var_linear_combination(a, b, sigma_x, sigma_y, rho):
    # Var(aX + bY) = a^2 sx^2 + b^2 sy^2 + 2 a b rho sx sy
    return a**2 * sigma_x**2 + b**2 * sigma_y**2 + 2 * a * b * rho * sigma_x * sigma_y


# Tracking error from section 4
fund_vol, bench_vol, rho = 0.15, 0.14, 0.98
tracking_error = np.sqrt(var_linear_combination(1, -1, fund_vol, bench_vol, rho))
print(f"Tracking error: {tracking_error:.2%}")

### Tracking error from simulated daily returns

In [ ]:
periods_per_year = 252
daily_fund_vol = fund_vol / np.sqrt(periods_per_year)
daily_bench_vol = bench_vol / np.sqrt(periods_per_year)

cov_daily = np.array(
    [
        [daily_fund_vol**2, rho * daily_fund_vol * daily_bench_vol],
        [rho * daily_fund_vol * daily_bench_vol, daily_bench_vol**2],
    ]
)
returns = rng.multivariate_normal(mean=[0.0003, 0.0003], cov=cov_daily, size=10 * periods_per_year)
fund_r, bench_r = returns[:, 0], returns[:, 1]

active_r = fund_r - bench_r
print(f"Fund vol (ann.):       {fund_r.std(ddof=1) * np.sqrt(periods_per_year):.2%}")
print(f"Benchmark vol (ann.):  {bench_r.std(ddof=1) * np.sqrt(periods_per_year):.2%}")
print(f"Tracking error (ann.): {active_r.std(ddof=1) * np.sqrt(periods_per_year):.2%}")

## 6. Visualizations

In [ ]:
rho_grid = np.linspace(-1, 1, 201)
sigma = 0.20

vol_sum = np.sqrt(var_linear_combination(0.5, 0.5, sigma, sigma, rho_grid))
vol_diff = np.sqrt(var_linear_combination(1, -1, sigma, sigma, rho_grid))

plt.figure(figsize=(8, 5))
plt.plot(rho_grid, vol_sum, label="50/50 portfolio  σ(0.5X + 0.5Y)")
plt.plot(rho_grid, vol_diff, label="hedged position  σ(X − Y)")
plt.axhline(sigma, color="grey", linestyle="--", label="single asset σ = 20%")
plt.title("Same formula, opposite wish: sums want low ρ, differences want high ρ")
plt.xlabel("correlation ρ")
plt.ylabel("annual volatility")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = np.arange(1.5, 13.5, 1)

axes[0].hist(independent_sum[:100_000], bins=bins, density=True, rwidth=0.85)
axes[0].set_title("Independent dice: X1 + X2")
axes[1].hist(glued_sum[:100_000], bins=bins, density=True, rwidth=0.85, color="tab:orange")
axes[1].set_title("Glued dice: 2X")

for ax in axes:
    ax.set_xlabel("sum")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("probability")
plt.suptitle("Same mean (7), different spread")
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. $\operatorname{Var}(X) = \operatorname{Var}(Y) = 4$. Which correlation $\rho$ makes $\operatorname{Var}(X-Y) = 0$?
2. Same variables, $\rho = -0.5$. Compute $\operatorname{Var}(X+Y)$ and $\operatorname{Var}(X-Y)$.

### Intermediate

3. A fund has 18% volatility, its benchmark 17%. What is the minimum correlation needed for a
   tracking error of at most 2%? Check your answer with `var_linear_combination`.
4. Interpret: a fund marketed as "active" reports a tracking error of 0.8% against the SMI.
   What does that say about the fund, and about its fee?

### Advanced

5. Show that $TE \ge |\sigma_F - \sigma_B|$ for any correlation, with equality only at $\rho = 1$.
   What does this imply for a fund whose volatility is far from its benchmark's?

## 8. Solutions / Checks

In [ ]:
# 1. Var(X - Y) = 2 * 4 * (1 - rho) = 0  ->  rho = 1
print("1. rho =", 1)

# 2.
print("2. Var(X+Y) =", var_linear_combination(1, 1, 2, 2, -0.5),
      "  Var(X-Y) =", var_linear_combination(1, -1, 2, 2, -0.5))

# 3. Solve 0.18^2 + 0.17^2 - 2 rho 0.18 0.17 <= 0.02^2 for rho
fund_vol, bench_vol, te_max = 0.18, 0.17, 0.02
rho_min = (fund_vol**2 + bench_vol**2 - te_max**2) / (2 * fund_vol * bench_vol)
print(f"3. rho_min = {rho_min:.4f}")
print(f"   check TE at rho_min: {np.sqrt(var_linear_combination(1, -1, fund_vol, bench_vol, rho_min)):.4f}")

**4.** A tracking error under 1% means the fund almost replicates the index (a "closet indexer").
Its active return is small and close to deterministic, so an active-management fee buys very
little beyond what an index fund delivers.

**5.** $TE^2 = \sigma_F^2 + \sigma_B^2 - 2\rho\sigma_F\sigma_B \ge \sigma_F^2 + \sigma_B^2 - 2\sigma_F\sigma_B = (\sigma_F - \sigma_B)^2$,
since $\rho \le 1$. Equality needs $\rho = 1$. A fund with 25% volatility against a 15% benchmark
has at least 10% tracking error, however correlated it is: different risk levels alone create
active risk.

## 9. Conclusion

- $E[X \pm Y] = E[X] \pm E[Y]$, always.
- $\operatorname{Var}(X \pm Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) \pm 2\operatorname{Cov}(X,Y)$: variances always add; only covariance carries the sign.
- Sums (portfolios) want low correlation: diversification.
- Differences (tracking error, hedges, spreads) want high correlation.
- Independent instruments make the worst hedge: the variance of the difference is the full sum.

Next: [02 Correlation and what it does not tell you](./02_correlation_and_what_it_does_not_tell_you.ipynb).